## CiFar 10 Dataset
### -> CNN implementation using Pytorch

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchmetrics

normalize using the CIFAR-10 mean & std for better convergence.
- This mean and std values are statistical obtained for CIfAR-10 

In [2]:
# CIFAR-10 mean & std for each channel (R, G, B)
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:20<00:00, 8.31MB/s] 


CNN Model

In [3]:
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super(CIFAR10_CNN, self).__init__()
        # Conv layers
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)   # 32x32x32
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # 32x32x64
        self.pool = nn.MaxPool2d(2, 2)                # halves dimensions

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1) # 16x16x128
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)# 16x16x128

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)

        x = x.view(-1, 128 * 8 * 8)  # flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CIFAR10_CNN()
print(model)

CIFAR10_CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)


Loss Function & Optimizer

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Model Training

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")
    

Epoch [1/10], Loss: 1.2312
Epoch [2/10], Loss: 0.7708
Epoch [3/10], Loss: 0.5791
Epoch [4/10], Loss: 0.4245
Epoch [5/10], Loss: 0.2896
Epoch [6/10], Loss: 0.1934
Epoch [7/10], Loss: 0.1372
Epoch [8/10], Loss: 0.1108
Epoch [9/10], Loss: 0.0948
Epoch [10/10], Loss: 0.0932


Evaluation

In [6]:
from torchmetrics.classification import MulticlassAccuracy

accuracy = MulticlassAccuracy(num_classes=10)

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        accuracy.update(preds, labels)

print(f"Test Accuracy: {accuracy.compute().item() * 100:.2f}%")

Test Accuracy: 76.01%
